In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

C:\Users\timer\AppData\Local\Temp\ipykernel_1636\2611793992.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [3]:
from langchain.tools import tool


@tool
def sql_query(query: str) -> str:
    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"


sql_query.invoke("SELECT * FROM Artist LIMIT 10")

"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]"

In [ ]:
from langchain.agents import create_agent

agent = create_agent(model="gpt-5-nano", tools=[sql_query])

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(
    content="Who is the most popular artist beginning with 'S' in this database?"
)

response = agent.invoke({"messages": [question]})

In [6]:
from pprint import pprint

pprint(response["messages"])

[HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?", additional_kwargs={}, response_metadata={}, id='94f1338f-94e0-418b-a06a-4a8f20164296'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 552, 'prompt_tokens': 142, 'total_tokens': 694, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EZtDRyTdEjIKDNb5NpXeiIwDchxa9', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a148c8-17bf-7e30-b44a-1ee202d41c0e-0', tool_calls=[{'name': 'sql_query', 'args': {'query': "SELECT name FROM artists WHERE name LIKE 'S%' ORDER BY popularity DESC LIMIT 1;"}, 'id': 'cal

In [7]:
print(response["messages"][-3].tool_calls[0]["args"]["query"])

SELECT a.Name FROM Artist a JOIN Album al ON al.ArtistId = a.ArtistId JOIN Track t ON t.AlbumId = al.AlbumId JOIN InvoiceLine il ON il.TrackId = t.TrackId WHERE a.Name LIKE 'S%' GROUP BY a.Name ORDER BY SUM(il.Quantity) DESC LIMIT 1;


In [9]:
print(response["messages"][-5].tool_calls[0]["args"]["query"])

PRAGMA table_info(Artist);


In [10]:
print(response["messages"][-7].tool_calls[0]["args"]["query"])

SELECT name FROM sqlite_master WHERE type='table';


In [13]:
question = HumanMessage(
    content="What information does the database provide (tables and content of each table)?"
)

response = agent.invoke({"messages": [question]})
print(response["messages"][-1].content)

Here’s what the database contains and some example content from each table. The data reflects a Chinook-like schema for a music store.

Tables in the database
- Album
- Artist
- Customer
- Employee
- Genre
- Invoice
- InvoiceLine
- MediaType
- Playlist
- PlaylistTrack
- Track

Details per table (columns, primary keys, and sample rows)

1) Album
- Columns: AlbumId (INTEGER, PK), Title (NVARCHAR(160)), ArtistId (INTEGER)
- Sample rows (first 5):
  - (1, 'For Those About To Rock We Salute You', 1)
  - (2, 'Balls to the Wall', 2)
  - (3, 'Restless and Wild', 2)
  - (4, 'Let There Be Rock', 1)
  - (5, 'Big Ones', 3)

2) Artist
- Columns: ArtistId (INTEGER, PK), Name (NVARCHAR(120))
- Sample rows (first 5):
  - (1, 'AC/DC')
  - (2, 'Accept')
  - (3, 'Aerosmith')
  - (4, 'Alanis Morissette')
  - (5, 'Alice In Chains')

3) Customer
- Columns: CustomerId (INTEGER, PK)
  - FirstName (NVARCHAR(40))
  - LastName (NVARCHAR(20))
  - Company (NVARCHAR(80))
  - Address (NVARCHAR(70))
  - City (NVARCHA

In [12]:
question = HumanMessage(
    content="What information does the table Invoice and InvoiceLine contain?"
)

response = agent.invoke({"messages": [question]})
print(response["messages"][-1].content)

Here is what each table contains, based on the schema you retrieved:

- Invoice
  - InvoiceId: INTEGER, NOT NULL, PRIMARY KEY
  - CustomerId: INTEGER, NOT NULL
  - InvoiceDate: DATETIME, NOT NULL
  - BillingAddress: NVARCHAR(70), NULL
  - BillingCity: NVARCHAR(40), NULL
  - BillingState: NVARCHAR(40), NULL
  - BillingCountry: NVARCHAR(40), NULL
  - BillingPostalCode: NVARCHAR(10), NULL
  - Total: NUMERIC(10,2), NOT NULL

- InvoiceLine
  - InvoiceLineId: INTEGER, NOT NULL, PRIMARY KEY
  - InvoiceId: INTEGER, NOT NULL
  - TrackId: INTEGER, NOT NULL
  - UnitPrice: NUMERIC(10,2), NOT NULL
  - Quantity: INTEGER, NOT NULL

Notes:
- Invoice is the header line for an invoice (one row per invoice) and includes the total amount.
- InvoiceLine represents the line items for an invoice (each row is an item on an invoice).
- The primary keys are InvoiceId (in Invoice) and InvoiceLineId (in InvoiceLine).
- The nullability shows most billing address fields are optional, while key fields and totals are

In [16]:
question = HumanMessage(
    content="Analyze which tables the databse contains and what data each table holds. Then, anser the question: How would you determine how popular an artist is?"
)

response = agent.invoke({"messages": [question]})
print(response["messages"][-1].content)

Here’s what I found about the database structure and data, followed by how to measure an artist’s popularity.

What tables exist and what data they hold
- Artist
  - Columns: ArtistId (PK), Name
  - What it stores: Artist identifiers and artist names.

- Album
  - Columns: AlbumId (PK), Title, ArtistId (FK to Artist)
  - What it stores: Albums with their titles and a link to the artist that created them.

- Track
  - Columns: TrackId (PK), Name, AlbumId (FK to Album), MediaTypeId (FK), GenreId (FK), Composer, Milliseconds, Bytes, UnitPrice
  - What it stores: Individual tracks, their metadata, and a link to the album they belong to.

- Genre
  - Columns: GenreId (PK), Name
  - What it stores: Music genres.

- MediaType
  - Columns: MediaTypeId (PK), Name
  - What it stores: Media type (e.g., MPEG, WAV, etc.) for tracks.

- Playlist
  - Columns: PlaylistId (PK), Name
  - What it stores: User-created playlists.

- PlaylistTrack
  - Columns: PlaylistId (FK to Playlist), TrackId (FK to Tra